# PCS5917 — Aula 7: análise das defesas

# Efeito das defesas: 45 casos finais
A análise usa 15 entradas de Arthur, 15 manuais originais de Roberto e 15 pedidos de código de Roberto. Os 15 casos de política inventada de Caio saem da análise. São 33 manuais e 12 PAIR; 315 respostas.

Recalculo o ASR e separo os 33 sucessos e as 12 falhas do baseline. As coletas de origem e os notebooks de ataques/julgamento estão na pasta pública do Drive indicada no README.


In [1]:
from pathlib import Path
from collections import Counter
import csv, json
RAIZ = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "README.md").is_file() and
                (p / "resultados/aula-07/analise-final/avaliados.csv").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra o notebook pela raiz do repositório ou pela pasta notebooks/.")
BASE = RAIZ / "resultados/aula-07/analise-final"

import csv
from collections import Counter

def ler_csv(p):
    with p.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def conferir_matriz(linhas, n):
    assert len(linhas) == n * 7
    assert len({x["id_resposta"] for x in linhas}) == len(linhas)
    casos = {x["id_caso"] for x in linhas}
    assert len(casos) == n
    condicoes = {"B","SR_EN","SR_PT","IA_EN","IA_PT","SRIA_EN","SRIA_PT"}
    for caso in casos:
        q = [x for x in linhas if x["id_caso"] == caso]
        assert len(q) == 7 and {x["condicao"] for x in q} == condicoes
        assert len({x["prompt_original"] for x in q}) == 1

def resumo_asr(linhas):
    saida = []
    baseline = {x["id_caso"]:x["rotulo_jev"] for x in linhas if x["condicao"] == "B"}
    for tipo in ["todos", "manual", "automatico"]:
        for cond in ["B","SR_EN","SR_PT","IA_EN","IA_PT","SRIA_EN","SRIA_PT"]:
            q = [x for x in linhas if x["condicao"] == cond and (tipo == "todos" or x["tipo"] == tipo)]
            c = Counter(x["rotulo_jev"] for x in q)
            den = c["sucesso"] + c["falha"]
            pares = [x for x in q if x["rotulo_jev"] in ["sucesso","falha"] and baseline.get(x["id_caso"]) in ["sucesso","falha"]]
            saida.append(dict(tipo=tipo, condicao=cond, total=len(q), sucessos=c["sucesso"], falhas=c["falha"],
                inconclusivos=c["inconclusivo"], denominador=den, asr_pct=round(100*c["sucesso"]/den,3) if den else None,
                pares_conclusivos=len(pares), sucesso_para_falha=sum(baseline[x["id_caso"]]=="sucesso" and x["rotulo_jev"]=="falha" for x in pares),
                falha_para_sucesso=sum(baseline[x["id_caso"]]=="falha" and x["rotulo_jev"]=="sucesso" for x in pares)))
    return saida

def imprimir_resumo(linhas):
    print("tipo          condição       S/n        ASR%    inconclusivos")
    for x in linhas:
        print(f'{x["tipo"]:13s} {x["condicao"]:10s} {x["sucessos"]:3d}/{x["denominador"]:<3d} {str(x["asr_pct"]):>9s} {x["inconclusivos"]:5d}')


In [2]:
dados = ler_csv(BASE / "avaliados.csv")
conferir_matriz(dados, 45)
calculado = resumo_asr(dados)
recebido = ler_csv(BASE / "resumo_asr.csv")
assert len(calculado) == len(recebido) == 21
for a,b in zip(calculado, recebido):
    for k in a:
        assert a[k] == b[k] if k in ["tipo","condicao"] else abs(float(a[k])-float(b[k])) < .00001
imprimir_resumo(calculado)


tipo          condição       S/n        ASR%    inconclusivos
todos         B           33/45     73.333     0
todos         SR_EN       27/45       60.0     0
todos         SR_PT       19/45     42.222     0
todos         IA_EN       10/45     22.222     0
todos         IA_PT       13/45     28.889     0
todos         SRIA_EN     14/45     31.111     0
todos         SRIA_PT     14/45     31.111     0
manual        B           24/33     72.727     0
manual        SR_EN       19/33     57.576     0
manual        SR_PT       13/33     39.394     0
manual        IA_EN        9/33     27.273     0
manual        IA_PT       12/33     36.364     0
manual        SRIA_EN     12/33     36.364     0
manual        SRIA_PT     13/33     39.394     0
automatico    B            9/12       75.0     0
automatico    SR_EN        8/12     66.667     0
automatico    SR_PT        6/12       50.0     0
automatico    IA_EN        1/12      8.333     0
automatico    IA_PT        1/12      8.333     0
automat

In [3]:
baseline = {x["id_caso"]:x["rotulo_jev"] for x in dados if x["condicao"] == "B"}
assert Counter(baseline.values()) == {"sucesso":33,"falha":12}
coortes = ler_csv(BASE / "coortes_baseline.csv")
for c in coortes:
    q = [x for x in dados if x["condicao"] == c["condicao"]]
    ss = sum(x["rotulo_jev"] == "sucesso" and baseline[x["id_caso"]] == "sucesso" for x in q)
    fs = sum(x["rotulo_jev"] == "sucesso" and baseline[x["id_caso"]] == "falha" for x in q)
    assert ss == int(c["permanecem_sucesso"]) and fs == int(c["novos_sucessos"])
    print(c["condicao"], "sucessos mantidos:", ss, "/33; novos sucessos:", fs, "/12")


B sucessos mantidos: 33 /33; novos sucessos: 0 /12
SR_EN sucessos mantidos: 25 /33; novos sucessos: 2 /12
SR_PT sucessos mantidos: 18 /33; novos sucessos: 1 /12
IA_EN sucessos mantidos: 10 /33; novos sucessos: 0 /12
IA_PT sucessos mantidos: 12 /33; novos sucessos: 1 /12
SRIA_EN sucessos mantidos: 13 /33; novos sucessos: 1 /12
SRIA_PT sucessos mantidos: 12 /33; novos sucessos: 2 /12


## Gráfico usado no relatório
![Efeito das defesas](../images/aula-07/efeito-defesas-baseline.png)

Cada painel tem seu denominador. As linhas tracejadas marcam os valores do baseline: 33 sucessos no painel (a) e 12 falhas no painel (b). O painel (b) conta as falhas mantidas; barras abaixo de 12 representam regressões. IA_EN bloqueia 23/33 sucessos e não introduz sucesso nas 12 falhas. O total final é 10/45 = 22,2%. As combinações empatam em 14/45 = 31,1%, mas têm transições diferentes. As escalas dos painéis são distintas e isso está indicado na legenda.
